In [26]:
%load_ext autoreload
%autoreload 2
%cd "/home/hew/python/contp"
%ls

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
/home/hew/python/contp
ckpt/  dataset/      dataset.zip  model/     script/  utils/
data/  dataset_bak/  generation/  README.md  temp/


/home/hew/anaconda3/envs/contp/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [96]:
t = 1.0
ood_fasta = f'/home/hew/python/TPNet/data/clustering/substrate_split/substrate_id{t}_df_split.fasta'
command = f'''
python ./script/predict.py --query_fasta {ood_fasta} --task substrate --out_dir ./temp/ood/ --save_dist
'''
print(command)


python ./script/predict.py --query_fasta /home/hew/python/TPNet/data/clustering/substrate_split/substrate_id1.0_df_split.fasta --task substrate --out_dir ./temp/ood/ --save_dist



In [97]:
import pandas as pd

from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MultiLabelBinarizer


def compute_label_metrics(preds, labels, classes=None):
    # preds: [[3], [0]], labels: [[3, 1], [0]]
    epoch_metrics = {}

    if classes is not None:
        mlb = MultiLabelBinarizer(classes=classes)
    else:
        mlb = MultiLabelBinarizer()

    mlb.fit(labels + preds)
    y_true = mlb.transform(labels)
    y_pred = mlb.transform(preds)

    # weighted
    weighted_precision, weighted_recall, weighted_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )

    # samples
    samples_precision, samples_recall, samples_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='samples', zero_division=0
    )

    # micro P/R/F1
    micro_precision, micro_recall, micro_f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='micro', zero_division=0
    )

    # sample-wise exact accuracy (subset accuracy)
    samples_acc = accuracy_score(y_true, y_pred)

    # micro accuracy (flattened into a binary classification problem)
    micro_acc = accuracy_score(y_true.ravel(), y_pred.ravel())

    # save
    epoch_metrics['weighted_precision'] = weighted_precision
    epoch_metrics['weighted_recall'] = weighted_recall
    epoch_metrics['weighted_f1'] = weighted_f1

    epoch_metrics['samples_precision'] = samples_precision
    epoch_metrics['samples_recall'] = samples_recall
    epoch_metrics['samples_f1'] = samples_f1
    epoch_metrics['samples_acc'] = samples_acc

    epoch_metrics['micro_precision'] = micro_precision
    epoch_metrics['micro_recall'] = micro_recall
    epoch_metrics['micro_f1'] = micro_f1
    epoch_metrics['micro_acc'] = micro_acc
    return epoch_metrics

In [117]:
t = 1.0
split_df = pd.read_csv(f'/home/hew/python/TPNet//data/clustering/substrate_split/substrate_id{t}_df_split.csv')
split_test_df = split_df[split_df['split'] == 'test']
split_test_df

,id,sequence,label,substrate,label_id,substrate_class,substrate_ids,substrate_num,split,length
5616,3.A.9.1.4,MALSMIRKSAAAPVRRAAAAPVVVRGRRVITFAAVDKAKVLEDVRS...,3.A.9.1,CHEBI:14911;protein,820,['amino acid:protein'],[2],1,test,117
5617,3.A.9.1.4,MASTARRKPVPEAMSEDXVEQEFGGDYXEASLEYIRALGPKKGAPF...,3.A.9.1,CHEBI:14911;protein,820,['amino acid:protein'],[2],1,test,299
5618,3.A.9.1.4,MATTSAPTSEPWTFDLVGQLRQKFGLGPENWDRFKGQAAEVPGADV...,3.A.9.1,CHEBI:14911;protein,820,['amino acid:protein'],[2],1,test,691
5619,3.A.9.1.4,MDQAPGPLGHLQRSFEHVGRGLASQLEQVEQAFVPMREGVTRWLER...,3.A.9.1,CHEBI:14911;protein,820,['amino acid:protein'],[2],1,test,365
5620,3.A.9.1.4,MGGELAERKASSASAGPRVVQMRREAIPAELLLVKEDPSKLPAGVL...,3.A.9.1,CHEBI:14911;protein,820,['amino acid:protein'],[2],1,test,137
...,...,...,...,...,...,...,...,...,...,...
43964,UniRef90_W9BE17,MPRIVLCCAAGMSTSMLVNKMKAEAQQRALALEIYAVAVAEFEREL...,4.A.3.2,CHEBI:6353;alpha-lactose,863,['sugar:disaccharide'],[11],1,test,102
43965,UniRef90_W9BFL5,MKIKAPDALLAAEVSRRGLMKTTAIGGLALASNALTLPFTRLSHAA...,5.A.3.3,CHEBI:10545;electron,882,['electron:electron'],[68],1,test,812
43970,UniRef90_X2C522,MTLRFPRFSQGLAQDPTTRRIWFGIATAHDFESHDDITEERLYQNI...,5.B.4.1,CHEBI:10545;electron,892,['electron:electron'],[68],1,test,734
43971,UniRef90_X2FCV1,MFMLNIILLIIPILLAVAFLTLIERKVLGYMQFRKGPNIVGPHGLL...,3.D.1.6,CHEBI:5584;hydron,827,['cation:proton'],[23],1,test,318


In [118]:
split_test_df['substrate_class'] = split_test_df['substrate_class'].map(eval)
split_test_df['main_class'] = split_test_df['substrate_class'].map(lambda x: set([item.split(':')[0] for item in x]))
split_out_df = df = pd.read_csv(f'./temp/ood/substrate_id{t}_df_split_pred.csv')
split_out_df['pred_class'] = split_out_df['substrate_top1'].map(lambda x: x.split(':')[0])
split_out_df

/home/hew/ipykernel_2205377/2854640889.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  split_test_df['substrate_class'] = split_test_df['substrate_class'].map(eval)
/home/hew/ipykernel_2205377/2854640889.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  split_test_df['main_class'] = split_test_df['substrate_class'].map(lambda x: set([item.split(':')[0] for item in x]))


,header,substrate_top1,substrate_top2,substrate_top3,pred_class
0,key_0,cation:proton,NaN,NaN,cation
1,key_1,amino acid:protein,NaN,NaN,amino acid
2,key_2,amino acid:protein,NaN,NaN,amino acid
3,key_3,amino acid:protein,NaN,NaN,amino acid
4,key_4,cation:proton,NaN,NaN,cation
...,...,...,...,...,...
13045,key_13045,sugar:monosaccharide,NaN,NaN,sugar
13046,key_13046,electron:electron,NaN,NaN,electron
13047,key_13047,electron:electron,NaN,NaN,electron
13048,key_13048,cation:proton,NaN,NaN,cation


In [119]:
split_out_df['pred_class'].unique()

array(['cation', 'amino acid', 'others', 'anion', 'nucleic acid',
       'electron', 'lipid', 'sugar'], dtype=object)

In [120]:
y_true_list = [list(y) for y in split_test_df['main_class'].tolist()]
y_pred_list = [[y] for y in split_out_df['pred_class'].tolist()]
y_true_list[:5], y_pred_list[:5]

([['amino acid'],
  ['amino acid'],
  ['amino acid'],
  ['amino acid'],
  ['amino acid']],
 [['cation'], ['amino acid'], ['amino acid'], ['amino acid'], ['cation']])

In [121]:
compute_label_metrics(y_pred_list, y_true_list)

{'weighted_precision': 0.9584694745670851,
 'weighted_recall': 0.9384938493849385,
 'weighted_f1': 0.9478835423096188,
 'samples_precision': 0.9587739463601532,
 'samples_recall': 0.9501085568326949,
 'samples_f1': 0.9529655172413793,
 'samples_acc': 0.9416091954022988,
 'micro_precision': 0.9587739463601532,
 'micro_recall': 0.9384938493849385,
 'micro_f1': 0.9485255098172997,
 'micro_acc': 0.986992337164751}

In [122]:
compute_label_metrics(y_pred_list, y_true_list)['samples_f1']

0.9529655172413793

In [123]:
'''
identity, samples_f1
1.00, 0.9529655172413793
0.99, 0.9066056560930297
0.95, 0.870546377501303
0.90, 0.8708733576239372
0.80, 0.8713702379375801
0.70, 0.8676657584014532
0.60, 0.8709393758167763
0.50, 0.8613770447841244
0.40, 0.8640907551642397
0.30, 0.8650959017524492
'''

'\nidentity, samples_f1\n1.00, 0.9529655172413793\n0.99, 0.9066056560930297\n0.95, 0.870546377501303\n0.90, 0.8708733576239372\n0.80, 0.8713702379375801\n0.70, 0.8676657584014532\n0.60, 0.8709393758167763\n0.50, 0.8613770447841244\n0.40, 0.8640907551642397\n0.30, 0.8650959017524492\n'